In [2]:
# Step 1: Import the libraries for K-Means
# Why this step comes first:
# Before we do any clustering, we need the tools for loading data,
# scaling features, running K-Means, and checking the cluster quality.

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [4]:
# Step 2: Load the dataset for K-Means
# Why this step comes after Step 1:
# We already imported the tools.
# Now we bring the data into the notebook so we can prepare it for clustering.

ALLData = pd.read_csv("All_features.csv", index_col="Date", parse_dates=["Date"]).dropna()

print("Shape:", ALLData.shape)
print("\nDate Range:", ALLData.index.min(), "to", ALLData.index.max())
print("\nNull Count:")
print(ALLData.isnull().sum())
print("\nDuplicates:", ALLData.index.duplicated().sum())

print("\nFirst 5 rows:")
print(ALLData.head())

Shape: (2342, 17)

Date Range: 2015-01-05 00:00:00 to 2025-12-30 00:00:00

Null Count:
HDFCBANK     0
NIFTY50      0
BANKNIFTY    0
SENSEX       0
INDIA_VIX    0
USDINR       0
SP500        0
NASDAQ100    0
DOWJONES     0
NIKKEI225    0
HANGSENG     0
GOLD         0
BRENT_OIL    0
ICICIBANK    0
SBIN         0
AXISBANK     0
KOTAKBANK    0
dtype: int64

Duplicates: 0

First 5 rows:
              HDFCBANK    NIFTY50    BANKNIFTY      SENSEX   INDIA_VIX  \
Date                                                                     
2015-01-05  500.999146  53.110001  1203.900024  218.399033  302.995667   
2015-01-06  483.090759  51.099998  1219.300049  214.999222  290.143066   
2015-01-07  482.703583  51.150002  1210.599976  215.626694  282.297943   
2015-01-08  485.898041  50.959999  1208.400024  220.155975  289.976135   
2015-01-09  479.121918  50.110001  1216.000000  222.620316  285.302460   

               USDINR       SP500   NASDAQ100      DOWJONES     NIKKEI225  \
Date               

In [10]:
# Step 3: Separate features for K-Means
# Why this step comes after Step 2:
# We already loaded and checked the data.
# Now we split the dataset into features and target.
# K-Means should use only features, not the target column.

X = ALLData.drop(columns=["HDFCBANK"])
y = ALLData["HDFCBANK"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())
print("\nTarget distribution:")
print(y.value_counts())

Feature matrix shape: (2342, 16)
Target shape: (2342,)

Feature columns:
['NIFTY50', 'BANKNIFTY', 'SENSEX', 'INDIA_VIX', 'USDINR', 'SP500', 'NASDAQ100', 'DOWJONES', 'NIKKEI225', 'HANGSENG', 'GOLD', 'BRENT_OIL', 'ICICIBANK', 'SBIN', 'AXISBANK', 'KOTAKBANK']

Target distribution:
505.909119     3
432.195435     3
682.151733     3
743.246643     3
539.474731     3
              ..
617.700500     1
593.084045     1
589.851501     1
593.581299     1
1246.000000    1
Name: HDFCBANK, Length: 2250, dtype: int64


In [12]:
# Step 4: Scale the features for K-Means
# Why this step comes after Step 3:
# K-Means uses distance calculations, so scaling is required.
# We keep Target separate and only scale the feature matrix.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled feature shape:", X_scaled.shape)
print("\nFirst 3 scaled rows:")
print(X_scaled[:3])

Scaled feature shape: (2342, 16)

First 3 scaled rows:
[[-0.76220081 -0.84989856 -1.62341545 -0.79566755 -1.4021852  -1.68042342
  -0.57897419 -1.07039607 -1.3813035  -1.20073474 -0.03380755 -0.39744838
  -1.13488866 -1.19079245 -1.11621039 -1.06872303]
 [-0.87632487 -0.82681351 -1.63874622 -0.82923069 -1.3944357  -1.69616381
  -0.63133345 -1.11556898 -1.39665918 -1.21426261 -0.09377138  0.12836009
  -1.20153716 -1.19932433 -1.16404411 -1.11261374]
 [-0.87348579 -0.83985515 -1.63591676 -0.84971736 -1.36473001 -1.67515932
  -0.63031487 -1.11972455 -1.37151569 -1.19672977 -0.04399017  0.2441343
  -1.20126567 -1.19095583 -1.17435848 -1.11702816]]


In [14]:
# Step 5: Find the best number of clusters
# Why this step comes after Step 4:
# We already scaled the features.
# Now we try different k values and check cluster quality.

inertia_list = []
silhouette_list = []
k_values = range(2, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia_list.append(km.inertia_)
    silhouette_list.append(silhouette_score(X_scaled, labels))

results_k = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertia_list,
    "silhouette": silhouette_list
})

print(results_k)

    k       inertia  silhouette
0   2  14088.882873    0.541964
1   3   9625.915681    0.484967
2   4   7196.179512    0.424846
3   5   5852.186844    0.448568
4   6   4906.326743    0.423178
5   7   4062.241454    0.431734
6   8   3494.873039    0.433490
7   9   2986.142346    0.454857
8  10   2710.984760    0.417659


In [16]:
# Step 6: Fit final K-Means model with k = 2
# Why this step comes after Step 5:
# We checked multiple k values and found k=2 has the best silhouette score.
# Now we train the final clustering model.

kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

ALLData["Cluster"] = clusters

print("Cluster counts:")
print(ALLData["Cluster"].value_counts())

print("\nFirst 10 rows with clusters:")
print(ALLData[["HDFCBANK", "Cluster"]].head(10))

Cluster counts:
0    1342
1    1000
Name: Cluster, dtype: int64

First 10 rows with clusters:
              HDFCBANK  Cluster
Date                           
2015-01-05  500.999146        0
2015-01-06  483.090759        0
2015-01-07  482.703583        0
2015-01-08  485.898041        0
2015-01-09  479.121918        0
2015-01-13  488.366577        0
2015-01-14  483.623230        0
2015-01-15  497.611115        0
2015-01-16  498.337128        0
2015-01-20  533.766724        0


In [18]:
# Step 7: Compare clusters with HDFCBANK behavior
# Why this step comes after Step 6:
# We already created cluster labels.
# Now we want to understand what each cluster means.

cluster_summary = ALLData.groupby("Cluster")["HDFCBANK"].agg(["count", "mean", "median", "std", "min", "max"])
print(cluster_summary)

print("\nCluster-wise Target Mean:")
print(ALLData.groupby("Cluster")["HDFCBANK"].mean())

print("\nCluster-wise Target Median:")
print(ALLData.groupby("Cluster")["HDFCBANK"].median())

         count        mean      median         std         min          max
Cluster                                                                    
0         1342  554.917440  533.994995  105.452119  301.889496   818.362488
1         1000  960.720539  975.115112  187.846513  622.948914  1316.175171

Cluster-wise Target Mean:
Cluster
0    554.917440
1    960.720539
Name: HDFCBANK, dtype: float64

Cluster-wise Target Median:
Cluster
0    533.994995
1    975.115112
Name: HDFCBANK, dtype: float64


In [24]:
# Step 8: Check target direction inside each cluster
# Why this step comes after Step 7:
# We already know the clusters have different HDFCBANK price levels.
# Now we check whether the clusters also differ in target direction.

if "Target" in ALLData.columns:
    cluster_target_summary = ALLData.groupby("Cluster")["Target"].agg(["count", "mean", "median", "std"])
    print(cluster_target_summary)

    print("\nCluster-wise target mean:")
    print(ALLData.groupby("Cluster")["Target"].mean())

    print("\nCluster-wise target median:")
    print(ALLData.groupby("Cluster")["Target"].median())
else:
    print("Target column not found in ALLData.")

Target column not found in ALLData.


In [26]:
# Step 9: Inspect columns and create a direction target if needed
# Why this step comes after Step 8:
# We discovered that ALLData does not contain a separate Target column.
# So we inspect columns and create a proper direction label from HDFCBANK.

print("Columns in ALLData:")
print(ALLData.columns.tolist())

if "Direction" not in ALLData.columns:
    ALLData["Direction"] = (ALLData["HDFCBANK"].shift(-1) > ALLData["HDFCBANK"]).astype(int)

print("\nDirection distribution:")
print(ALLData["Direction"].value_counts(dropna=False))

print("\nFirst 10 rows with Direction:")
print(ALLData[["HDFCBANK", "Direction"]].head(10))

Columns in ALLData:
['HDFCBANK', 'NIFTY50', 'BANKNIFTY', 'SENSEX', 'INDIA_VIX', 'USDINR', 'SP500', 'NASDAQ100', 'DOWJONES', 'NIKKEI225', 'HANGSENG', 'GOLD', 'BRENT_OIL', 'ICICIBANK', 'SBIN', 'AXISBANK', 'KOTAKBANK', 'Cluster']

Direction distribution:
1    1212
0    1130
Name: Direction, dtype: int64

First 10 rows with Direction:
              HDFCBANK  Direction
Date                             
2015-01-05  500.999146          0
2015-01-06  483.090759          0
2015-01-07  482.703583          1
2015-01-08  485.898041          0
2015-01-09  479.121918          1
2015-01-13  488.366577          0
2015-01-14  483.623230          1
2015-01-15  497.611115          1
2015-01-16  498.337128          1
2015-01-20  533.766724          0


In [28]:
# Step 10: Compare clusters with Direction
# Why this step comes after Step 9:
# We now have a proper direction label.
# This lets us see whether each cluster tends to be bullish or bearish.

cluster_direction_summary = ALLData.groupby("Cluster")["Direction"].agg(["count", "mean", "sum"])
print(cluster_direction_summary)

print("\nCluster-wise direction mean:")
print(ALLData.groupby("Cluster")["Direction"].mean())

print("\nCluster-wise direction proportion of 1:")
print(ALLData.groupby("Cluster")["Direction"].mean().round(4))

print("\nCross-tab of Cluster vs Direction:")
print(pd.crosstab(ALLData["Cluster"], ALLData["Direction"], normalize="index"))

         count      mean  sum
Cluster                      
0         1342  0.516393  693
1         1000  0.519000  519

Cluster-wise direction mean:
Cluster
0    0.516393
1    0.519000
Name: Direction, dtype: float64

Cluster-wise direction proportion of 1:
Cluster
0    0.5164
1    0.5190
Name: Direction, dtype: float64

Cross-tab of Cluster vs Direction:
Direction         0         1
Cluster                      
0          0.483607  0.516393
1          0.481000  0.519000


In [30]:
# Step 11: Give the clusters a simple regime label
# Why this step comes after Step 10:
# We already checked that direction rates are very similar.
# Now we summarize the cluster meaning in a human-readable way.

cluster_summary = ALLData.groupby("Cluster").agg(
    count=("HDFCBANK", "size"),
    hdfc_mean=("HDFCBANK", "mean"),
    direction_mean=("Direction", "mean")
)

cluster_summary["regime"] = np.where(
    cluster_summary["hdfc_mean"] > cluster_summary["hdfc_mean"].median(),
    "Higher-price regime",
    "Lower-price regime"
)

print(cluster_summary)

print("\nInterpretation:")
print("Cluster 0 and Cluster 1 mainly separate lower-price and higher-price regimes.")
print("However, the next-day direction rate is almost the same in both clusters.")
print("So K-Means is useful for regime description, but not strong for direction prediction.")

         count   hdfc_mean  direction_mean               regime
Cluster                                                        
0         1342  554.917440        0.516393   Lower-price regime
1         1000  960.720539        0.519000  Higher-price regime

Interpretation:
Cluster 0 and Cluster 1 mainly separate lower-price and higher-price regimes.
However, the next-day direction rate is almost the same in both clusters.
So K-Means is useful for regime description, but not strong for direction prediction.


In [32]:
# Step 12: Final conclusion for K-Means
# Why this step comes after Step 11:
# We already interpreted the clusters and compared them with direction.
# Now we write the final project conclusion.

print("K-MEANS PROJECT SUMMARY")
print("-----------------------")
print("Number of clusters:", ALLData["Cluster"].nunique())
print("\nCluster counts:")
print(ALLData["Cluster"].value_counts())

print("\nCluster interpretation:")
print("Cluster 0 = Lower-price regime")
print("Cluster 1 = Higher-price regime")

print("\nDirection analysis:")
print("Cluster 0 bullish rate:", round(ALLData.groupby("Cluster")["Direction"].mean().iloc[0], 4))
print("Cluster 1 bullish rate:", round(ALLData.groupby("Cluster")["Direction"].mean().iloc[1], 4))

print("\nFinal conclusion:")
print("K-Means was useful for detecting market regimes in the HDFCBANK feature space.")
print("However, the clusters did not show a strong difference in next-day direction.")
print("So K-Means is better for regime understanding than for direct direction prediction.")

K-MEANS PROJECT SUMMARY
-----------------------
Number of clusters: 2

Cluster counts:
0    1342
1    1000
Name: Cluster, dtype: int64

Cluster interpretation:
Cluster 0 = Lower-price regime
Cluster 1 = Higher-price regime

Direction analysis:
Cluster 0 bullish rate: 0.5164
Cluster 1 bullish rate: 0.519

Final conclusion:
K-Means was useful for detecting market regimes in the HDFCBANK feature space.
However, the clusters did not show a strong difference in next-day direction.
So K-Means is better for regime understanding than for direct direction prediction.
